In [1]:
import requests
from bs4 import BeautifulSoup
import json
import time
import random
import re

def scrape_dsa_posts(subreddit="learnprogramming", search_term="DSA"):
    """
    Scrape DSA related posts from a specific subreddit
    """
    try:
        # URL for searching DSA in the specified subreddit
        url = f"https://old.reddit.com/r/{subreddit}/search?q={search_term}&restrict_sr=on&sort=relevance&t=all"
        
        # Headers to mimic browser request
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
        }
        
        print(f"Fetching DSA posts from r/{subreddit}...")
        response = requests.get(url, headers=headers, timeout=15)
        
        if response.status_code == 200:
            html_content = response.text
            soup = BeautifulSoup(html_content, "html.parser")
            
            # Search results are in divs with class "search-result search-result-link"
            # Based on the HTML screenshot you provided
            search_results = soup.select("div.search-result.search-result-link")
            
            if not search_results:
                # Try alternative selectors if the first doesn't work
                search_results = soup.select(".search-result-link")
            
            dsa_posts = []
            
            print(f"\n🔹 DSA Related Posts from r/{subreddit}:")
            
            for i, result in enumerate(search_results, start=1):
                try:
                    # Find the post title - it's in an <a> element with class "search-title"
                    title_elem = result.select_one("a.search-title") or result.select_one("a.may-blank.search-title")
                    
                    if not title_elem:
                        # Try alternative selector based on your screenshot
                        title_elem = result.select_one("a.may-blank")
                    
                    if title_elem:
                        title = title_elem.text.strip()
                        
                        # Get the post URL
                        href = title_elem.get("href", "")
                        
                        # Fix the URL if it's a relative path
                        if href and not href.startswith("http"):
                            href = f"https://www.reddit.com{href}" if not href.startswith('/') else f"https://www.reddit.com{href}"
                        
                        # Look for post metadata
                        meta_div = result.select_one("div.search-result-meta")
                        upvotes = "Unknown"
                        comments = "Unknown"
                        author = "Unknown"
                        post_date = "Unknown"
                        
                        # Try to extract post stats (upvotes, comments)
                        stats_text = ""
                        if meta_div:
                            stats_text = meta_div.text
                        else:
                            # Try to find scores elsewhere
                            score_elem = result.select_one(".score.unvoted")
                            if score_elem:
                                upvotes = score_elem.text.strip()
                            
                            # Find comments count
                            comments_elem = result.select_one("a.comments")
                            if comments_elem:
                                comments = comments_elem.text.strip()
                        
                        # Parse stats text if we found it
                        if stats_text:
                            # Extract upvotes
                            upvotes_match = re.search(r'(\d+) points', stats_text)
                            if upvotes_match:
                                upvotes = upvotes_match.group(1)
                            
                            # Extract comments
                            comments_match = re.search(r'(\d+) comments', stats_text)
                            if comments_match:
                                comments = comments_match.group(1)
                            
                            # Extract submission time
                            time_match = re.search(r'submitted (\d+ \w+ ago|\w+ ago)', stats_text)
                            if time_match:
                                post_date = time_match.group(1)
                            
                            # Extract author
                            author_match = re.search(r'by (\w+)', stats_text)
                            if author_match:
                                author = author_match.group(1)
                        
                        # Check for "Resource" tag
                        resource_tag = result.select_one("span.linkflairlabel")
                        resource_type = resource_tag.text.strip() if resource_tag else "None"
                        
                        # Print and save post info
                        print(f"{i}. {title}")
                        print(f"   Link: {href}")
                        print(f"   Upvotes: {upvotes} | Comments: {comments}")
                        print(f"   Posted: {post_date} by {author}")
                        print(f"   Type: {resource_type}\n")
                        
                        dsa_posts.append({
                            "title": title,
                            "url": href,
                            "upvotes": upvotes,
                            "comments": comments,
                            "author": author,
                            "post_date": post_date,
                            "resource_type": resource_type
                        })
                        
                except Exception as post_error:
                    print(f"Error processing result {i}: {post_error}")
                    continue
            
            print(f"Total DSA posts found: {len(dsa_posts)}")
            
            # Save to file
            filename = f"{subreddit}_dsa_posts.json"
            with open(filename, "w") as f:
                json.dump(dsa_posts, f, indent=4)
            
            print(f"Results saved to {filename}")
            
            return dsa_posts
            
        else:
            print(f"Failed to retrieve the webpage. Status code: {response.status_code}")
            
    except requests.exceptions.RequestException as e:
        print(f"Error during request: {e}")
    except Exception as e:
        print(f"An error occurred: {e}")
        
    return []

def extract_post_content(post_url, headers):
    """Extract the content and comments from a DSA post"""
    try:
        # Add a small random delay to avoid rate limiting
        time.sleep(random.uniform(1, 3))
        
        response = requests.get(post_url, headers=headers, timeout=15)
        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html.parser")
            
            # Extract post content - in a div with class "md"
            post_content_elem = soup.select_one("div.md")
            content = post_content_elem.text.strip() if post_content_elem else "No content found"
            
            # Extract code blocks if any
            code_blocks = []
            if post_content_elem:
                code_elements = post_content_elem.select("pre > code")
                for code_elem in code_elements:
                    code_blocks.append(code_elem.text)
            
            # Extract comments
            comments = []
            comment_elements = soup.select("div.comment")
            for i, comment_elem in enumerate(comment_elements[:10]):  # Limit to first 10 comments
                comment_text_elem = comment_elem.select_one("div.md")
                author_elem = comment_elem.select_one("a.author")
                
                if comment_text_elem:
                    author = author_elem.text if author_elem else "Unknown"
                    comments.append({
                        "author": author,
                        "text": comment_text_elem.text.strip()
                    })
            
            return {
                "content": content,
                "code_blocks": code_blocks,
                "comments": comments
            }
    except Exception as e:
        print(f"Error extracting content from {post_url}: {e}")
    
    return {"content": "", "code_blocks": [], "comments": []}

def create_dsa_helper():
    """Create a comprehensive DSA helper by scraping multiple subreddits"""
    dsa_resources = {}
    
    # List of subreddits to scrape for DSA content
    subreddits = ["learnprogramming", "cscareerquestions", "algorithms", "datastructures"]
    
    for subreddit in subreddits:
        print(f"\n{'='*50}")
        print(f"Scraping r/{subreddit} for DSA posts")
        print(f"{'='*50}")
        
        posts = scrape_dsa_posts(subreddit)
        dsa_resources[subreddit] = posts
        
        # Avoid hitting rate limits
        if subreddit != subreddits[-1]:
            delay = random.uniform(5, 10)
            print(f"Waiting {delay:.2f} seconds before next subreddit...")
            time.sleep(delay)
    
    # Save combined results
    with open("dsa_helper_resources.json", "w") as f:
        json.dump(dsa_resources, f, indent=4)
    
    print("\nDSA Helper resources compiled successfully!")
    return dsa_resources

if __name__ == "__main__":
    # To scrape just one subreddit (e.g., learnprogramming)
    # posts = scrape_dsa_posts("learnprogramming")
    
    # To create a comprehensive DSA helper with resources from multiple subreddits
    # create_dsa_helper()
    
    # Default: Scrape DSA posts from r/learnprogramming
    posts = scrape_dsa_posts("learnprogramming")
    
    # Uncomment to extract full content for the first few posts (max 5)
    """
    if posts:
        print("\nExtracting detailed content for the top posts...")
        headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
        }
        
        for i, post in enumerate(posts[:5]):  # Limit to first 5 posts
            print(f"Processing post {i+1}/5: {post['title']}")
            details = extract_post_content(post['url'], headers)
            post.update(details)
            time.sleep(random.uniform(2, 4))  # Be gentle with the server
            
        # Save detailed results
        with open("dsa_detailed_resources.json", "w") as f:
            json.dump(posts, f, indent=4)
            
        print("Detailed DSA resources saved to dsa_detailed_resources.json")
    """

Fetching DSA posts from r/learnprogramming...

🔹 DSA Related Posts from r/learnprogramming:
1. Learning DSA from scratch : The Ultimate Guide
   Link: https://old.reddit.com/r/learnprogramming/comments/14e52ia/learning_dsa_from_scratch_the_ultimate_guide/
   Upvotes: 424 | Comments: 82
   Posted: 1 year ago by Distinct_Expert_
   Type: Resource

2. Insights from an ex-Googler who has taught 1000s of Engineers about DSA interviews
   Link: https://old.reddit.com/r/learnprogramming/comments/1gpt0fz/insights_from_an_exgoogler_who_has_taught_1000s/
   Upvotes: 435 | Comments: 25
   Posted: 3 months ago by rahulkpandey
   Type: Resource

3. Guys who do leetcode/DSA everyday, how did you create that habit, whats your secret?
   Link: https://old.reddit.com/r/learnprogramming/comments/1i4ttys/guys_who_do_leetcodedsa_everyday_how_did_you/
   Upvotes: 83 | Comments: 50
   Posted: 1 month ago by vignank
   Type: None

4. DSA Study Material: I have been prepping for interviews for about 5 months 

In [10]:
import requests
from bs4 import BeautifulSoup
url = f"https://old.reddit.com/r/btechtards/search?q=DSA&restrict_sr=on&sort=relevance&t=all"
        
        # Headers to mimic browser request
headers = {
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
            'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
            'Accept-Language': 'en-US,en;q=0.5',
        }
        
print(f"Fetching DSA posts from r/btechtards...")
response = requests.get(url, headers=headers, timeout=15)
        
if response.status_code == 200:
    html_content = response.text
    soup = BeautifulSoup(html_content, "html.parser")
    search_results = soup.select("div.search-result.search-result-link")
    #print(search_results)
    for i, post in enumerate(search_results[:25], start=1):  
        # Extract post title
        title = post.find("a", class_="search-title").text.strip()

        # Extract post link
        post_link = post.find("a", class_="search-title")["href"]

        # Extract post score
        score_element = post.find("span", class_="search-score")
        score = score_element.text.strip() if score_element else "N/A"

        # Extract number of comments
        comments_element = post.find("a", class_="search-comments")
        comments = comments_element.text.strip() if comments_element else "0 comments"

        print(f"{i}. {title}")
        print(f" {post_link}")
        print(f" Score: {score} | {comments}\n")

Fetching DSA posts from r/btechtards...
1. DSA is wayyyyyyyy to difficult
 https://old.reddit.com/r/Btechtards/comments/1ifzk86/dsa_is_wayyyyyyyy_to_difficult/
 Score: 144 points | 56 comments

2. 19f need a DSA buddyy
 https://old.reddit.com/r/Btechtards/comments/1htzh1j/19f_need_a_dsa_buddyy/
 Score: 275 points | 170 comments

3. Google Girl Hackathon DSA Round – How Did It Go for You?
 https://old.reddit.com/r/Btechtards/comments/1it5vr2/google_girl_hackathon_dsa_round_how_did_it_go_for/
 Score: 24 points | 286 comments

4. Complete Competitive Programming & DSA guide that i followed during my placement
 https://old.reddit.com/r/Btechtards/comments/1bniob6/complete_competitive_programming_dsa_guide_that_i/
 Score: 666 points | 186 comments

5. someone want to learn the dsa or ml or backend ?
 https://old.reddit.com/r/Btechtards/comments/1i41qkq/someone_want_to_learn_the_dsa_or_ml_or_backend/
 Score: 103 points | 115 comments

6. Looking for DSA partner
 https://old.reddit.com/r/Btec

In [22]:
import requests
from bs4 import BeautifulSoup

subreddit = "btechtards"
url = f"https://old.reddit.com/r/{subreddit}/search?q=DSA&restrict_sr=on&sort=relevance&t=all"

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
    'Accept-Language': 'en-US,en;q=0.5',
}

print(f"Fetching DSA posts from r/btechtards...")

response = requests.get(url, headers=headers, timeout=15)

if response.status_code == 200:
    html_content = response.text
    soup = BeautifulSoup(html_content, "html.parser")
    search_results = soup.select("div.search-result.search-result-link")

    print(f"\n🔹 Top DSA Posts from r/{subreddit}:\n")

    for i, post in enumerate(search_results[:25], start=1): 
        title = post.find("a", class_="search-title").text.strip()

        post_link = post.find("a", class_="search-title")["href"]
        new_post_link = post_link.replace("old.reddit.com", "www.reddit.com") #this is so that the final links direct to new reddit

        score_element = post.find("span", class_="search-score")
        score = score_element.text.strip() if score_element else "N/A"

        comments_element = post.find("a", class_="search-comments")
        comments = comments_element.text.strip() if comments_element else "0 comments"

        print(f"{i}. {title}")
        print(f"{new_post_link}")
        print(f"Score: {score} | {comments}\n")

else:
    print("Failed to fetch data. Status Code:", response.status_code)


Fetching DSA posts from r/btechtards...

🔹 Top DSA Posts from r/btechtards:

1. DSA is wayyyyyyyy to difficult
https://www.reddit.com/r/Btechtards/comments/1ifzk86/dsa_is_wayyyyyyyy_to_difficult/
Score: 145 points | 💬 56 comments

2. 19f need a DSA buddyy
https://www.reddit.com/r/Btechtards/comments/1htzh1j/19f_need_a_dsa_buddyy/
Score: 276 points | 💬 170 comments

3. Google Girl Hackathon DSA Round – How Did It Go for You?
https://www.reddit.com/r/Btechtards/comments/1it5vr2/google_girl_hackathon_dsa_round_how_did_it_go_for/
Score: 24 points | 💬 286 comments

4. Complete Competitive Programming & DSA guide that i followed during my placement
https://www.reddit.com/r/Btechtards/comments/1bniob6/complete_competitive_programming_dsa_guide_that_i/
Score: 666 points | 💬 186 comments

5. someone want to learn the dsa or ml or backend ?
https://www.reddit.com/r/Btechtards/comments/1i41qkq/someone_want_to_learn_the_dsa_or_ml_or_backend/
Score: 106 points | 💬 115 comments

6. Looking for DSA p